In [ ]:
%pip install -U transformers
%pip install -U datasets
%pip install wikipedia
%pip install flytekit
%pip install codecarbon
%pip install peft bitsandbytes accelerate
%pip install numexpr


In [ ]:
# CHANGE: giảm phân mảnh GPU allocator (giúp tránh OOM do fragmentation)
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# Agent

In [ ]:
from datasets import load_dataset
import json
import re
import numexpr
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
)
from datasets import Dataset
import pandas as pd

# QLORA:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)  # CHANGE: QLORA
from transformers import BitsAndBytesConfig  # CHANGE: QLORA


# 1. TOOLS IMPLEMENTATION

In [ ]:
import wikipedia

wikipedia.set_lang("vi")


def Calculator(query: str) -> str:
    """NumExpr based calculator for numerical expressions."""
    try:
        tool_response = str(numexpr.evaluate(query))
    except Exception as e:
        print(f"Calculator error: {e}")
        tool_response = f"Error: failed to calculate {query}."
    return tool_response


def WikipediaRetriever(query: str, length: int = 300) -> str:
    """
    Placeholder Wikipedia retriever.
    In production, this would call Wikipedia API or use a retrieval system.
    """

    results = wikipedia.page(query, auto_suggest=False)

    summary = wikipedia.summary[:length]


In [ ]:
import wikipedia

page = wikipedia.page("thể tích", auto_suggest=False)  # auto_suggest=True by default
print(page.summary)


# 2. TOOLS METADATA (Fixed format for Qwen)


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "Calculator",
            "description": "Evaluate a numeric expression given as a string, using + - * / and parentheses.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A numeric expression, e.g. '4+5*(2-1)'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "Wikipedia_retriever",
            "description": "Return relevant Wikipedia document text for a query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"},
                    "k": {
                        "type": "integer",
                        "description": "Number of sentences in the topic",
                        "default": 5,
                    },
                },
                "required": ["query"],
            },
        },
    },
]


# 3. UTILITY FUNCTIONS


In [ ]:
def extract_answer(response):
    """Extract answer from response."""
    try:
        answer = response.split("\nAnswer:")[1].strip()
    except Exception:
        answer = response
    return answer


def extract_tool_usage(llm_text: str):
    """Extract tool usage patterns like ToolName[arg] from model output."""
    try:
        action = llm_text.split("\nAction:")[1].split("\nRationale:")[0]
    except Exception:
        action = llm_text
    return re.findall(r"\w+\[[^\[\]]+\]", action)


def parse_tool_call_from_text(text: str):
    """Parse <tool_call>{...}</tool_call> and return dict or None."""
    m = re.search(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", text, flags=re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(1))
    except Exception as e:
        print(f"Failed to parse tool_call JSON: {e}")
        return None


# 4. DATA PREPARATION


## 4.1 5CD-AI/Vietnamese-395k-meta-math-MetaMathQA-gg-translated

In [ ]:
def prepare_data(ds, tokenizer, max_length=512, batch_size=1000):
    """Process dataset efficiently using map."""

    if isinstance(ds, pd.DataFrame):
        ds = Dataset.from_pandas(ds)

    def format_and_tokenize(examples):
        formatted_texts = []
        for query, response in zip(examples["query_vi"], examples["response_vi"]):
            messages = [
                {
                    "role": "system",
                    "content": "Please reason step by step, and give your final answer.",
                },
                {"role": "user", "content": query},
                {"role": "assistant", "content": response},
            ]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False, tools=tools
            )
            formatted_texts.append(text)

        #
        # tokenized = tokenizer(
        #     formatted_texts,
        #     padding="max_length",
        #     truncation=True,
        #     max_length=max_length,
        # )
        tokenized = tokenizer(
            formatted_texts,
            padding=False,  # CHANGE: dùng padding động để giảm VRAM (thay "max_length")
            truncation=True,
            max_length=max_length,  # CHANGE: giữ theo tham số truyền vào (sẽ gọi với 384)
        )

        # tokenized["labels"] = tokenized["input_ids"] # Đừng tự gán labels — để collator tự tạo sau khi pad => Collator sẽ pad input_ids rồi tự tạo labels từ input_ids đã pad, nên không còn lệch.
        return tokenized

    processed_dataset = ds.map(
        format_and_tokenize,
        batched=True,
        batch_size=batch_size,
        num_proc=4,
        remove_columns=ds.column_names,
        desc="Tokenizing dataset",
    )

    return processed_dataset


In [ ]:
ds = load_dataset("5CD-AI/Vietnamese-395k-meta-math-MetaMathQA-gg-translated")["train"]


In [ ]:
model_name = "Qwen/Qwen2.5-Math-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


### No use tool

In [ ]:
query = "1 + 1 is equal to ?"  # assumed defined earlier
response = "1 + 1 is equal to 2"

messages = [
    {
        "role": "system",
        "content": "Please reason step by step, and give your final answer.",
    },
    {"role": "user", "content": query},
    {
        "role": "assistant",
        "content": "I'll compute the numeric expression first.",
        "tool_calls": [
            {
                # your template supports tool_call.function or tool_call.name
                "function": {"name": "Calculator", "arguments": {"expression": "1+1"}}
            }
        ],
    },
    # tool response recorded in the dataset
    {"role": "tool", "content": "2"},
    # assistant final response (teacher / human-provided)
    {"role": "assistant", "content": response},
]

# Format for model input using your tokenizer template:
formatted = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=False, tools=tools
)
print(formatted)  # inspect the result to be sure it matches expectations


### Use tool

In [ ]:
query = "Hãy tính 0.5*0.2/0.00000008"  # assumed defined earlier
response = "0.5*0.2/0.00000008 = 1,250,000"

messages = [
    {
        "role": "system",
        "content": "Please reason step by step, and give your final answer.",
    },
    {"role": "user", "content": query},
    {
        "role": "assistant",
        "content": "Đầu tiên, nhân 0.5 với 0.2:",
        "tool_calls": [
            {
                # your template supports tool_call.function or tool_call.name
                "function": {
                    "name": "Calculator",
                    "arguments": {"expression": "0.5*0.2"},
                }
            }
        ],
    },
    # tool response recorded in the dataset
    {"role": "tool", "content": "0.1"},
    # assistant final response (teacher / human-provided)
    {
        "role": "assistant",
        "content": "Tiếp theo, Chia cho 0.00000008 :",
        "tool_calls": [
            {
                # your template supports tool_call.function or tool_call.name
                "function": {
                    "name": "Calculator",
                    "arguments": {"expression": "0.1/0.0000008"},
                }
            }
        ],
    },
    {"role": "tool", "content": "1,250,000"},
    {"role": "assistant", "content": response},
]

# Format for model input using your tokenizer template:
formatted = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=False, tools=tools
)
print(formatted)


In [ ]:
ds_split = ds.train_test_split(test_size=0.2, shuffle=True)
train_ds = ds_split["train"].to_pandas()
eval_ds = ds_split["test"].to_pandas()
eval_ds = prepare_data(eval_ds, tokenizer, max_length=512)
train_ds = prepare_data(train_ds, tokenizer, max_length=512)


In [ ]:
# ds_split = ds.train_test_split(test_size=0.05, shuffle=True)
# ds = ds_split["test"]
# ds_split = ds.train_test_split(test_size=0.2, shuffle=True)
# train_ds = ds_split["train"].to_pandas()
# eval_ds = ds_split["test"].to_pandas()
# # eval_ds = prepare_data(eval_ds, tokenizer)
# # train_ds = prepare_data(train_ds, tokenizer)

# eval_ds = prepare_data(eval_ds, tokenizer, max_length=384)
# train_ds = prepare_data(train_ds, tokenizer, max_length=384)


## 4.2 VNHSGE

# 5. CONFIGURATION


In [ ]:
class Config:
    # Training settings
    OUTPUT_DIR = "./results"
    LOGGING_DIR = "./logs"
    EPOCHS = 1
    BATCH_SIZE = 2
    PER_DEVICE_EVAL_BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 32
    LEARNING_RATE = 2e-4
    WEIGHT_DECAY = 0.05
    MAX_GRAD_NORM = 0.4
    WARMUP_RATIO = 0.03
    MAX_LEN = 512  # CHANGE: thêm trường để dùng thống nhất

    # Evaluation and saving
    SAVE_STRATEGY = "steps"
    EVAL_STRATEGY = "steps"
    EVAL_STEPS = 700
    SAVE_STEPS = 700
    SAVE_TOTAL_LIMIT = 2
    LOGGING_STEPS = 10

    # Mixed precision
    FP16 = False
    BF16 = True

    # Other settings
    LR_SCHEDULER_TYPE = "constant_with_warmup"
    REPORT_TO = "codecarbon"
    LOAD_BEST_MODEL_AT_END = True
    METRIC_FOR_BEST_MODEL = "eval_loss"
    GREATER_IS_BETTER = False
    EVAL_ACCUMULATION_STEPS = 16


In [ ]:
config = Config()


# 6. TOOL USE AGENT


In [ ]:
class ToolUseAgent:
    def __init__(
        self,
        model,
        tokenizer,
        tools_metadata=None,
        generation_cfg=None,
        device=None,  # FIXED: Added device parameter
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.tools = tools_metadata or []

        # FIXED: Properly set device
        if device is None:
            self.device = next(model.parameters()).device
        else:
            self.device = torch.device(device)

        # Generation defaults
        self.generation_cfg = generation_cfg or {
            "max_new_tokens": 256,
            "do_sample": False,
            "temperature": 0.0,
            "top_p": 0.95,
        }

    def invoke_tool(self, func_str: str) -> str:
        """Invoke a tool based on the function string."""
        if not isinstance(func_str, str) or "[" not in func_str:
            return "Error: invalid tool invocation format."

        tool_name = func_str.split("[", 1)[0]
        query = func_str.split("[", 1)[1].rsplit("]", 1)[0]

        # FIXED: Corrected tool name matching
        if tool_name.lower() in ("calculator",):
            return Calculator(query)
        elif tool_name.lower() in (
            "wikipedia_retriever",
            "wikipediasearch",
            "wikipediaretriever",
        ):
            return WikipediaRetriever(query)
        else:
            return f"Error: tool `{tool_name}` not found."

    def call_llm(self, conversations: list, add_generation_prompt=True):
        """Call the language model with the conversation history."""
        # Render chat template
        prompt_text = self.tokenizer.apply_chat_template(
            conversations,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            tools=self.tools if self.tools else None,
        )

        # Tokenize to tensors
        encoded = self.tokenizer(prompt_text, return_tensors="pt")
        input_ids = encoded["input_ids"].to(self.device)
        attention_mask = encoded.get("attention_mask", None)
        if attention_mask is not None:
            attention_mask = attention_mask.to(self.device)

        # Generation arguments
        gen_kwargs = dict(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=self.generation_cfg.get("max_new_tokens", 256),
            do_sample=self.generation_cfg.get("do_sample", False),
            temperature=self.generation_cfg.get("temperature", 0.0),
            top_p=self.generation_cfg.get("top_p", 0.95),
            pad_token_id=self.tokenizer.eos_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )

        # Generate
        with torch.no_grad():
            outputs = self.model.generate(**gen_kwargs)

        # Decode
        generated = outputs[0, input_ids.shape[-1] :].cpu().numpy()
        decoded = self.tokenizer.decode(generated, skip_special_tokens=True).strip()
        return decoded

    def inference(self, question: str, prompt_type="default"):
        """Run inference with tool usage capability."""
        system_prompt = (
            "Please reason step by step, and put your final answer within \\boxed{}."
        )

        conversations = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ]

        llm_response = self.call_llm(conversations, add_generation_prompt=True)
        conversations.append({"role": "assistant", "content": llm_response})

        # Check for tool calls
        tool_call = parse_tool_call_from_text(llm_response)
        tools_simple = extract_tool_usage(llm_response)

        # FIXED: Added max iterations to prevent infinite loops
        max_iterations = 5
        iteration = 0

        while (
            tool_call is not None or len(tools_simple) > 0
        ) and iteration < max_iterations:
            iteration += 1

            if tool_call is not None:
                name = tool_call.get("name")
                args = tool_call.get("arguments", {})

                # Construct invocation string
                if "expression" in args:
                    invocation = f"{name}[{args['expression']}]"
                elif "query" in args:
                    invocation = f"{name}[{args['query']}]"
                else:
                    invocation = f"{name}[{json.dumps(args)}]"

                tool_res = self.invoke_tool(invocation)
                tool_text = f"Response from tool {invocation}: {tool_res}"

                # FIXED: Use 'user' role instead of mixing roles
                conversations.append({"role": "tool", "content": tool_text})
                llm_response = self.call_llm(conversations, add_generation_prompt=True)
                conversations.append({"role": "assistant", "content": llm_response})
            else:
                # Handle simple tool patterns
                for func in tools_simple:
                    tool_res = self.invoke_tool(func)
                    tool_text = f"Response from tool {func}: {tool_res}"
                    conversations.append({"role": "user", "content": tool_text})

                llm_response = self.call_llm(conversations, add_generation_prompt=True)
                conversations.append({"role": "assistant", "content": llm_response})

            # Check for more tool calls
            tool_call = parse_tool_call_from_text(llm_response)
            tools_simple = extract_tool_usage(llm_response)

        return conversations, llm_response

    def train(self, train_dataset, eval_dataset, cfg):
        """Train the model."""
        try:
            self.model.config.use_cache = False
        except Exception:
            pass
        # Enable gradient checkpointing
        self.model.gradient_checkpointing_enable()

        # Data collator
        data_collator = DataCollatorForLanguageModeling(
            tokenizer=self.tokenizer, mlm=False
        )

        # Training arguments
        training_args = TrainingArguments(
            output_dir=cfg.OUTPUT_DIR,
            num_train_epochs=cfg.EPOCHS,
            per_device_train_batch_size=cfg.BATCH_SIZE,
            per_device_eval_batch_size=cfg.PER_DEVICE_EVAL_BATCH_SIZE,
            gradient_accumulation_steps=cfg.GRADIENT_ACCUMULATION_STEPS,
            learning_rate=cfg.LEARNING_RATE,
            weight_decay=cfg.WEIGHT_DECAY,
            warmup_ratio=cfg.WARMUP_RATIO,
            lr_scheduler_type=cfg.LR_SCHEDULER_TYPE,
            fp16=cfg.FP16,
            eval_strategy=cfg.EVAL_STRATEGY,
            eval_steps=cfg.EVAL_STEPS,
            save_strategy=cfg.SAVE_STRATEGY,
            save_steps=cfg.SAVE_STEPS,
            save_total_limit=cfg.SAVE_TOTAL_LIMIT,
            logging_steps=cfg.LOGGING_STEPS,
            logging_dir=cfg.LOGGING_DIR,
            load_best_model_at_end=cfg.LOAD_BEST_MODEL_AT_END,
            metric_for_best_model=cfg.METRIC_FOR_BEST_MODEL,
            remove_unused_columns=False,
            gradient_checkpointing=True,
            report_to=cfg.REPORT_TO,
            tf32=True,  # CHANGE: bật TF32 để tăng tốc
        )

        # Trainer
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            processing_class=self.tokenizer,
            data_collator=data_collator,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )

        # CHANGE: debug peak memory trước/sau train
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        # Train
        trainer.train()
        trainer.save_model(training_args.output_dir)


# 7. Training


In [ ]:
model_name = "Qwen/Qwen2.5-Math-1.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# CHANGE: đảm bảo có pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# CHANGE: tối ưu bộ nhớ khi train
try:
    model.config.use_cache = False
except:
    pass
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

# (Tùy chọn) nếu đã cài flash-attn 2
try:
    model.config.attn_implementation = "flash_attention_2"
    print("Use flash_attention_2")
except:
    print("Do not use flash_attention_2")
    pass

agent = ToolUseAgent(model, tokenizer, tools, device=device)


In [ ]:
agent.train(train_ds, eval_ds, config)
